# Shreve Week 10 — Multidimensional Stock Model

**Week 10** — Multidimensional Stock Model

This notebook teaches **multidimensional stock model** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **Multi-asset GBM** | Ch. 5.3 |
| 2 | **Correlation & Cholesky** | Ch. 5.3 |
| 3 | **Portfolio of options** | Ch. 5.3 |
| 4 | **Diversification** | Ch. 5.3 |
| — | **Problem set** | Ch. 5 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 5 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Multi-Asset GBM

\(dS_i = r S_i\, dt + \sigma_i S_i\, dB_i\) under \(Q\), with correlated Brownian drivers.

Discounted prices \(e^{-rt}S_t\) are martingales under \(Q\).

**Shreve Ch. 5.3:** multidimensional stock model.


In [ ]:
def multi_gbm(S0, sigmas, rho_matrix, r, T, n_steps, seed=100):
    m = len(S0)
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    L = np.linalg.cholesky(rho_matrix)
    S = np.array(S0, dtype=float)
    paths = [S.copy()]
    for _ in range(n_steps):
        Z = rng.standard_normal(m)
        dB = np.sqrt(dt) * (L @ Z)
        S = S * np.exp((r - 0.5*np.array(sigmas)**2)*dt + np.array(sigmas)*dB)
        paths.append(S.copy())
    return np.array(paths)

S0 = [100, 50, 80]
sigmas = [0.2, 0.3, 0.25]
rho = np.array([[1, 0.5, 0.3], [0.5, 1, 0.4], [0.3, 0.4, 1]])
paths = multi_gbm(S0, sigmas, rho, 0.05, 1.0, 252)
print(f"Final prices: {paths[-1]}")


---
# Part 2 — Correlation via Cholesky

Correlation matrix \(\rho\) → Cholesky \(L\) with \(\rho = LL^\top\). Independent \(Z\) → \(B = LZ\).

**Shreve Ch. 5.3:** constructing correlated BM.


In [ ]:
rho2 = np.array([[1, 0.6], [0.6, 1]])
L = np.linalg.cholesky(rho2)
rng = np.random.default_rng(101)
Z = rng.standard_normal((100_000, 2))
B = (L @ Z.T).T
print(f"Target ρ=0.6, simulated = {np.corrcoef(B[:,0], B[:,1])[0,1]:.4f}")


---
# Part 3 — Basket / Exchange Options

Price \(e^{-rT} E^Q[(S_1(T) + S_2(T) - K)^+]\) by Monte Carlo.

**Shreve Ch. 5.3:** multi-asset derivatives.


In [ ]:
rng = np.random.default_rng(102)
n = 300_000
T, r, K = 1.0, 0.05, 150
S1_T = paths[-1, 0]  # reuse single path endpoint — resim for MC
# proper MC
dt = T/252
L = np.linalg.cholesky(rho2)
payoffs = []
for _ in range(n):
    Z = rng.standard_normal(2)
    dB = np.sqrt(T) * (L @ Z)
    S1 = 100*np.exp((r-0.5*0.2**2)*T + 0.2*dB[0])
    S2 = 50*np.exp((r-0.5*0.3**2)*T + 0.3*dB[1])
    payoffs.append(max(S1+S2-K, 0))
basket_price = np.exp(-r*T) * np.mean(payoffs)
print(f"Basket call (S1+S2-K)+ price ≈ {basket_price:.2f}")


---
# Part 4 — Diversification Effect

Portfolio variance \(\sigma_p^2 = w^\top \Sigma w\) with covariance \(\Sigma_{ij} = \rho_{ij}\sigma_i\sigma_j\).

Lower average correlation → lower portfolio vol for fixed weights.

**Shreve Ch. 5.3:** correlation risk.


In [ ]:
w = np.array([0.4, 0.3, 0.3])
cov = np.outer(sigmas, sigmas) * rho
port_var = w @ cov @ w
port_vol = np.sqrt(port_var)
avg_vol = np.mean(sigmas)
print(f"Portfolio vol = {port_vol:.4f}, avg asset vol = {avg_vol:.4f}")
print(f"Diversification ratio = {avg_vol/port_vol:.2f}")


**Your turn:** How does basket option price change as \(\rho_{12}\) increases?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Write the multidimensional GBM under \(Q\) with correlation matrix \(\rho\).

2. Price a spread option on \(S_1 - S_2\) by simulation.

3. Show discounted \(S_i\) are martingales under \(Q\).

4. Find portfolio weights minimizing variance for two assets.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(dS_i = rS_i dt + \sigma_i S_i \sum_j L_{ij} dW_j\) with \(LL^\top=\rho\).

**2.** MC: \(e^{-rT} E[(S_1-S_2-K)^+]\); correlation affects spread distribution.

**3.** \(d(e^{-rt}S_i) = e^{-rt}\sigma_i S_i dB_i^Q\) — no drift under \(Q\).

**4.** \(w^* = \Sigma^{-1}1 / (1^\top\Sigma^{-1}1)\) for min-var portfolio.

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 5 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
